In [1]:
# Install required packages if needed
!pip install scikit-learn matplotlib seaborn joblib pandas numpy chardet rdkit statsmodels

_homeDir = r"Box/clg/M.Tech/Paper Work/CuPyridineQSAR"

import pandas as pd
import numpy as np
import csv
import statsmodels.api as sm

# Input library for feature extraction, and feature engineering
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator

# Model parameters for Linear Regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, RobustScaler, Binarizer
from sklearn.feature_selection import RFE, SelectKBest, f_regression

# Plotting and visualization
import matplotlib.pyplot as plt
import joblib

import os
from datetime import datetime
import warnings
import chardet
from io import StringIO
import requests
warnings.filterwarnings('ignore')

Defaulting to user installation because normal site-packages is not writeable


C:\Users\mohan\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


**Data exploration**

Import curated literature data input

In [2]:
df = pd.read_csv(_homeDir + '/Data/00_base_data.csv')
print(f"Base data shape: {df.shape}")
print(f"Base data Information: {df.info()}")
#print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'Box/clg/M.Tech/Paper Work/CuPyridineQSAR/Data/00_base_data.csv'

**Data pre-processing**

In [ ]:
df_clean = df.dropna()
print("Cleaned data size: ", df_clean.shape)
print("Cleaned data info: ", df_clean.info())

# Extract useful feature and output only
select_column = ['L1', 'L2', 'L3', 'pIC50']

clean_data = df_clean[select_column]
clean_data.to_csv(_homeDir + '/Data/00_clean_data.csv', index=False)

**Feature extraction**

In this stage two operations are performed

Extracting descriptor values from rdkit library

In [ ]:
df_clean = pd.read_csv(_homeDir + '/Data/00_clean_data.csv')
print(f"Base data shape: {df_clean.shape}")
print(f"Base data Information: {df_clean.info()}")

# Define descriptor names
L1_descriptor_names = ['L1_MolWt', 'L1_MolLogP', 'L1_NumHDonors', 'L1_NumHAcceptors', 'L1_TPSA', 'L1_NumRotatableBonds', 'L1_NumAromaticRings', 'L1_NumAliphaticRings', 'L1_FractionCSP3', 'L1_HeavyAtomCount']
L2_descriptor_names = ['L2_MolWt', 'L2_MolLogP', 'L2_NumHDonors', 'L2_NumHAcceptors', 'L2_TPSA', 'L2_NumRotatableBonds', 'L2_NumAromaticRings', 'L2_NumAliphaticRings', 'L2_FractionCSP3', 'L2_HeavyAtomCount']
L3_descriptor_names = ['L3_MolWt', 'L3_MolLogP', 'L3_NumHDonors', 'L3_NumHAcceptors', 'L3_TPSA', 'L3_NumRotatableBonds', 'L3_NumAromaticRings', 'L3_NumAliphaticRings', 'L3_FractionCSP3', 'L3_HeavyAtomCount']

def extract_descriptors(smiles):
  mol = Chem.MolFromSmiles(smiles)
  if mol:
    descriptors = [
      Descriptors.MolWt(mol),
      Descriptors.MolLogP(mol),
      Descriptors.NumHDonors(mol),
      Descriptors.NumHAcceptors(mol),
      Descriptors.TPSA(mol),
      Descriptors.NumRotatableBonds(mol),
      Descriptors.NumAromaticRings(mol),
      Descriptors.NumAliphaticRings(mol),
      Descriptors.FractionCSP3(mol),
      Descriptors.HeavyAtomCount(mol),
    ]
    return descriptors
  else:
    return None

# Extract descriptors from Ligand 1 - L1
#print(f"Extract descriptors for L1...")
descriptors_data = {name: [] for name in L1_descriptor_names}

for smiles in df_clean['L1']:
  Descriptors_values = extract_descriptors(smiles)
  if Descriptors_values:
    for i, names in enumerate(L1_descriptor_names):
      descriptors_data[names].append(Descriptors_values[i])

  else:
    for names in L1_descriptor_names:
      descriptors_data[names].append(np.nan)

for name in L1_descriptor_names:
  df_clean[name] = descriptors_data[name]


# Extract descriptors from Ligand 2 - L2
#print(f"Extract descriptors for L1...")
descriptors_data = {name: [] for name in L2_descriptor_names}

for smiles in df_clean['L2']:
  Descriptors_values = extract_descriptors(smiles)
  if Descriptors_values:
    for i, names in enumerate(L2_descriptor_names):
      descriptors_data[names].append(Descriptors_values[i])

  else:
    for names in L2_descriptor_names:
      descriptors_data[names].append(np.nan)

for name in L2_descriptor_names:
  df_clean[name] = descriptors_data[name]

# Extract descriptors from Ligand 3 - L3
#print(f"Extract descriptors for L3...")
descriptors_data = {name: [] for name in L3_descriptor_names}

for smiles in df_clean['L3']:
  Descriptors_values = extract_descriptors(smiles)
  if Descriptors_values:
    for i, names in enumerate(L3_descriptor_names):
      descriptors_data[names].append(Descriptors_values[i])

  else:
    for names in L3_descriptor_names:
      descriptors_data[names].append(np.nan)

for name in L3_descriptor_names:
  df_clean[name] = descriptors_data[name]

#Again do data cleaning - remove empty values:
df_clean = df_clean.dropna()
print("Post desriptor extraction and clean-up, Unique number of features: ", df_clean.shape)

#Write to output file.
df_clean.to_csv(_homeDir + '/Data/00_feature_data.csv', index=False)

**Feature Engineering**

Data normalization

In [ ]:
df = pd.read_csv(_homeDir + '/Data/00_feature_data.csv')

# Configuration dictionary for various data inputs and its transform/normalization
normalize_config = {
    # Robust Scaling for continuous variables with outliers
    'L1_MolWt': {'method': 'robust_scale'},
    'L1_MolLogP': {'method': 'robust_scale'},
    'L1_TPSA': {'method': 'robust_scale'},
    'L1_FractionCSP3': {'method': 'robust_scale'},

    'L2_MolWt': {'method': 'robust_scale'},
    'L2_MolLogP': {'method': 'robust_scale'},
    'L2_TPSA': {'method': 'robust_scale'},
    'L2_FractionCSP3': {'method': 'robust_scale'},

    'L3_MolWt': {'method': 'robust_scale'},
    'L3_MolLogP': {'method': 'robust_scale'},
    'L3_TPSA': {'method': 'robust_scale'},
    'L3_FractionCSP3': {'method': 'robust_scale'},

    # Binarization for sparse count data
    'L1_NumHDonors': {'method': 'binarize', 'threshold': 0},
    'L2_NumHDonors': {'method': 'binarize', 'threshold': 0},
    'L2_NumHDonors': {'method': 'binarize', 'threshold': 0},

    # One-hot encoding for low cardinal count data
    'L1_NumHAcceptors': {'method': 'standard_scale'},
    'L1_NumRotatableBonds': {'method': 'standard_scale'},
    'L1_NumAromaticRings': {'method': 'standard_scale'},

    'L2_NumHAcceptors': {'method': 'standard_scale'},
    'L2_NumRotatableBonds': {'method': 'standard_scale'},
    'L2_NumAromaticRings': {'method': 'standard_scale'},

    'L3_NumHAcceptors': {'method': 'standard_scale'},
    'L3_NumRotatableBonds': {'method': 'standard_scale'},
    'L3_NumAromaticRings': {'method': 'standard_scale'},

    # Standard scaling for continuous variables
    'L1_HeavyAtomCount': {'method': 'standard_scale'},
    'L2_HeavyAtomCount': {'method': 'standard_scale'},
    'L3_HeavyAtomCount': {'method': 'standard_scale'},

    # Already this is effective representation. No normalization technique required
    'L1_NumAliphaticRings': {'method': 'none'},
    'L2_NumAliphaticRings': {'method': 'none'},
    'L3_NumAliphaticRings': {'method': 'none'}
}

# Apply transform/normalization tehcnique and write to file
for column, config in normalize_config.items():
  if config['method'] == 'robust_scale':
    # Robust scaling
    scaler = RobustScaler()
    scaled_values = scaler.fit_transform(df[[column]])
    df[column] = scaled_values

  elif config['method'] == 'standard_scale':
    # Standard scaling
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(df[[column]])
    df[column] = scaled_values

  elif config['method'] == 'binarize':
    # Binarization
    threshold = config.get('threshold', 0)
    binarizer = Binarizer(threshold=threshold)
    binary_values = binarizer.fit_transform(df[[column]])
    df[column] = binary_values.astype(int)

# Drop SMILES columns L1, L2, L3
df = df.drop(columns=['L1', 'L2', 'L3'])

df.to_csv(_homeDir + '/Data/00_normalized_data.csv', index=False)

**Model Building - Linear Regression**

Chose best feature base on feature selection methods like Statisticsl, SelectKBest, RFE, etc

In [ ]:
# Train Test split and fit linear regression model
df = pd.read_csv(_homeDir + '/Data/00_normalized_data.csv')

# Step # 1. Feature and Target selection
X = df.drop('pIC50', axis=1)
y = df['pIC50']  # Target variable

# Step # 2. Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results = {}

# 3.a Lasso feature selection
#print("Running Lasso feature selection...")
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train, y_train)
lasso_features = X_train.columns[lasso.coef_ != 0].tolist()

lr_model = LinearRegression()
lr_model.fit(X_train[lasso_features], y_train)
score = lr_model.score(X_test[lasso_features], y_test)

results['Lasso'] = {
    'score': score,
    'features': lasso_features,
    'num_features': len(lasso_features)
}
#print(f"Lasso method - R2 Score: {score:.6f}, Features: {len(lasso_features)}")

# 3.b SelectKBest feature selection
#print("Running SelectKBest feature selection...")
selector = SelectKBest(score_func=f_regression, k=10) # Use 10 best features as in your previous example
selector.fit(X_train, y_train)
kbest_features = X_train.columns[selector.get_support()].tolist()

lr_model = LinearRegression()
lr_model.fit(X_train[kbest_features], y_train)
score = lr_model.score(X_test[kbest_features], y_test)
results['SelectKBest'] = {
    'score': score,
    'features': kbest_features,
    'num_features': len(kbest_features)
}
#print(f"SelectKBest method - R2 Score: {score:.6f}, Features: {len(kbest_features)}")

# 3.c. RFE feature selection
#print("Running RFE feature selection...")
estimator = LinearRegression()
selector = RFE(estimator, n_features_to_select=10, step=1)
selector.fit(X_train, y_train)
rfe_features = X_train.columns[selector.support_].tolist()

lr_model = LinearRegression()
lr_model.fit(X_train[rfe_features], y_train)
score = lr_model.score(X_test[rfe_features], y_test)

results['RFE'] = {
  'score': score,
  'features': rfe_features,
  'num_features': len(rfe_features)
}
#print(f"RFE method - R2 Score: {score:.6f}, Features: {len(rfe_features)}")

# 3.d. Statistical method (p-value based selection)
#print("Running Statistical feature selection...")
X_train_const = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_const).fit()
# Get features with p-value < 0.05
significant_features = model.pvalues[model.pvalues < 0.05].index.tolist()
if 'const' in significant_features:
    significant_features.remove('const')

lr_model = LinearRegression()
lr_model.fit(X_train[significant_features], y_train)
score = lr_model.score(X_test[significant_features], y_test)

results['statistical'] = {
  'score': score,
  'features': significant_features,
  'num_features': len(significant_features)
}
#print(f"Statistical method - R2 Score: {score:.6f}, Features: {len(significant_features)}")
    
# Step # 3.e. Train LR model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
score = lr_model.score(X_test, y_test)

results['All Features'] = {
  'score': score,
  'features': X_train.columns.tolist(),
  'num_features': len(X_train.columns)
}
#print(f"All Features - R2 Score: {score:.6f}, Features: {len(X_train.columns)}")

# Find the best method
best_method = max(results.items(), key=lambda x: x[1]['score'])
print(f"\nBest method: {best_method[0]} with R2 Score: {best_method[1]['score']:.6f}")
print(f"Number of features selected: {best_method[1]['num_features']}")
print(f"Features: {best_method[1]['features']}")

**Test model with k-fold cross validation**

In [ ]:
from sklearn.model_selection import KFold

# Initialize arrays to store results
r2_scores = []
rmse_scores = []
mae_scores = []

# Manual cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    r2_scores.append(r2_score(y_test, y_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae_scores.append(mean_absolute_error(y_test, y_pred))

# Calculate final metrics
r2_cv = np.mean(r2_scores)
rmse_cv = np.mean(rmse_scores)
mae_cv = np.mean(mae_scores)
std_dev = np.std(r2_scores)

#Write to output file
with open(_homeDir + '/Results/results.csv', 'w', newline='') as file:
  writer = csv.writer(file)
  
  #Write header
  writer.writerow(["Model", "Variant", "R2(CV)", "RMSE", "MAE", "Std. Dev", "Rank"])

  #Write data output
  writer.writerow(["Linear Regression", best_method[0], f"{r2_cv:.6f}", f"{rmse_cv:.6f}", f"{mae_cv:.6f}", f"{std_dev:.6f}", 1])